# Intro

Este proceso se realiza luego de la extracción de los artículos utilizando **Scopus Search API** con la query `AFFIL(AFFILCOUNTRY(ECUADOR))`, la cual busca la parte del país de la dirección de un autor y devuelve dichos artículos.

Se obtuvó un total de `69392 articulos` desde 1920 hasta 2026, para reducir el consumo de la API de Scopus se ha guardado la `response`, en su forma original, en MongoDB donde cada artículo es un documento en la colección y es el archivo JSON con el cual se trabaja en este notebook.

En [Scopus Search View](https://dev.elsevier.com/sc_search_views.html) se encuentra la documentación de que atributos se pueden obtener para cada artículo.

<figure style="text-align: center;">
  <img src="imgs/tbl_limites_API_Scopus.jpeg" alt="Tabla descriptiva de limites de las APIs de Scopus" width="900"/>
  <figcaption>Tabla descriptiva de limites de las APIs de Scopus</figcaption>
</figure>

# Bibliotecas

In [52]:
import pandas as pd
import numpy as np

In [53]:
# Mostrar todas las columnas al imprimir el DataFrame
pd.set_option("display.max_columns", None)

# Carga de Datos

In [54]:
# Datos raw
df_raw_articles = pd.read_json("data/ETL_centinela.raw_data_search_articles.json")

In [55]:
print(f"Datos cargados: {df_raw_articles.shape}\n")
df_raw_articles.head(3)

Datos cargados: (69392, 39)



,_id,dc:identifier,@_fa,affiliation,article-number,authkeywords,author,author-count,citedby-count,control_fields,dc:creator,dc:description,dc:title,eid,freetoread,freetoreadLabel,fund-acr,fund-sponsor,link,openaccess,openaccessFlag,prism:aggregationType,prism:coverDate,prism:coverDisplayDate,prism:doi,prism:eIssn,prism:issueIdentifier,prism:pageRange,prism:publicationName,prism:url,prism:volume,pubmed-id,source-id,subtype,subtypeDescription,prism:issn,prism:isbn,fund-no,pii
0,{'$oid': '6911f77771b47a74ac0f6ac6'},SCOPUS_ID:85090049295,true,"[{'@_fa': 'true', 'affiliation-url': 'https://...",1806527,catastrophic health expenditure | chronic dise...,"[{'@_fa': 'true', '@seq': '1', 'author-url': '...","{'@limit': '100', '$': '8'}",23,"{'source': 'scopus_search_api', 'inserted_at':...",Londoño Agudelo E.,Background: Hypertension requires life-long me...,Out-of-pocket expenditure for hypertension car...,2-s2.0-85090049295,"{'value': [{'$': 'all'}, {'$': 'publisherfullg...","{'value': [{'$': 'All Open Access'}, {'$': 'Go...",DG DEVCO,Directorate-General for International Cooperat...,"[{'@_fa': 'true', '@ref': 'self', '@href': 'ht...",1,True,Journal,2020-12-31,31 December 2020,10.1080/16549716.2020.1806527,16549880,1,None,Global Health Action,https://api.elsevier.com/content/abstract/scop...,13,32867605.0,1.750015e+10,ar,Article,NaN,NaN,NaN,NaN
1,{'$oid': '6911f77771b47a74ac0f6ac7'},SCOPUS_ID:85093859087,true,"[{'@_fa': 'true', 'affiliation-url': 'https://...",NaN,Banks | Competition | Market concentration,"[{'@_fa': 'true', '@seq': '1', 'author-url': '...","{'@limit': '100', '$': '3'}",9,"{'source': 'scopus_search_api', 'inserted_at':...",Solano J.,Purpose: Banks are institutions that inject mo...,Intra-banking competition in Ecuador: new evid...,2-s2.0-85093859087,"{'value': [{'$': 'all'}, {'$': 'publisherfullg...","{'value': [{'$': 'All Open Access'}, {'$': 'Go...",NaN,NaN,"[{'@_fa': 'true', '@ref': 'self', '@href': 'ht...",1,True,Journal,2020-12-30,30 Dec 2020,10.1108/JEFAS-05-2019-0081,22180648,50,295-319,Journal of Economics Finance and Administrativ...,https://api.elsevier.com/content/abstract/scop...,25,NaN,2.110038e+10,ar,Article,20771886,NaN,NaN,NaN
2,{'$oid': '6911f77771b47a74ac0f6ac8'},SCOPUS_ID:85098991959,true,"[{'@_fa': 'true', 'affiliation-url': 'https://...",NaN,Carbon dioxide capture | Circular economy | En...,"[{'@_fa': 'true', '@seq': '1', 'author-url': '...","{'@limit': '100', '$': '5'}",28,"{'source': 'scopus_search_api', 'inserted_at':...",Ruiz C.,We report a highly efficient system for CO2 ca...,Sustainable and Negative Carbon Footprint Soli...,2-s2.0-85098991959,NaN,NaN,NaN,NaN,"[{'@_fa': 'true', '@ref': 'self', '@href': 'ht...",0,False,Journal,2020-12-28,28 December 2020,10.1021/acssuschemeng.0c07093,21680485,51,19003-19012,ACS Sustainable Chemistry and Engineering,https://api.elsevier.com/content/abstract/scop...,8,NaN,2.110025e+10,ar,Article,NaN,NaN,NaN,NaN


# Exploración de Datos

## Seleccionar Columnas Clave

Dentro de todos los atributos de un artículo, los más importantes son:
- `dc:identifier`: identificador de Scopus (Scopus ID)
- `dc:title`: titulo
- `dc:description`: abstract
- `authkeywords`: palabras clave del autor
- `citedby-count`: cantidad de citaciones
- `prism:coverDate`: fecha de publicación (YYYY-MM-DD)
- `affiliation`: lista de afiliaciones
- `author`: lista de autores

In [56]:
key_columns = [
    "dc:identifier",
    "dc:title",
    "dc:description",
    "authkeywords",
    "citedby-count",
    "prism:coverDate",
    "affiliation",
    "author",
]

In [57]:
df_articles = df_raw_articles[key_columns].copy()
df_articles

,dc:identifier,dc:title,dc:description,authkeywords,citedby-count,prism:coverDate,affiliation,author
0,SCOPUS_ID:85090049295,Out-of-pocket expenditure for hypertension car...,Background: Hypertension requires life-long me...,catastrophic health expenditure | chronic dise...,23,2020-12-31,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,SCOPUS_ID:85093859087,Intra-banking competition in Ecuador: new evid...,Purpose: Banks are institutions that inject mo...,Banks | Competition | Market concentration,9,2020-12-30,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,SCOPUS_ID:85098991959,Sustainable and Negative Carbon Footprint Soli...,We report a highly efficient system for CO2 ca...,Carbon dioxide capture | Circular economy | En...,28,2020-12-28,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
3,SCOPUS_ID:85102966055,Sentiment Analysis Tool for Spanish Tweets in ...,The huge amount of textual information that ex...,Azure Text Analytics | IBM Watson NLU | Lexico...,2,2020-12-24,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
4,SCOPUS_ID:85098243025,Maxillaria purpureo-nigra (Orchidaceae: Maxill...,"Maxillaria purpureo-nigra Zambrano, Carnevali ...",Identification key | Maxillaria prunina | Morm...,3,2020-12-24,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
...,...,...,...,...,...,...,...,...
69387,SCOPUS_ID:105010830178,Multicriteria Analysis Based on GIS for the Po...,The present research is entitled “MULTICRITERI...,Artisanal aquaculture | GIS (Geographic Inform...,0,2026-01-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
69388,SCOPUS_ID:105010826250,Tourism Ecosystems New Challenges,The global challenges in terms of climate chan...,Challenges | Governance | Sustainability | Tou...,0,2026-01-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
69389,SCOPUS_ID:105010814584,Flexible Textile Antenna for IoT Applications ...,"In this paper, the design of a flexible textil...",Characteristic modes theory | conductive mater...,0,2026-01-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
69390,SCOPUS_ID:105008406214,Extended Grüss type inequalities for generaliz...,The main motivation of this paper is to establ...,fractional integrals | Inequalities | kernel |...,0,2026-01-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


## Estructura General

In [58]:
df_articles.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69392 entries, 0 to 69391
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   dc:identifier    69392 non-null  object
 1   dc:title         69391 non-null  object
 2   dc:description   65889 non-null  object
 3   authkeywords     58228 non-null  object
 4   citedby-count    69392 non-null  int64 
 5   prism:coverDate  69392 non-null  object
 6   affiliation      69130 non-null  object
 7   author           69341 non-null  object
dtypes: int64(1), object(7)
memory usage: 4.2+ MB


## Valores Nulos

In [59]:
# Nulos por columna
df_articles.isnull().sum().sort_values(ascending=False)

authkeywords       11164
dc:description      3503
affiliation          262
author                51
dc:title               1
dc:identifier          0
prism:coverDate        0
citedby-count          0
dtype: int64

In [60]:
# Nulos por fila
df_articles.isnull().any(axis=1).sum()

np.int64(11702)

Es decir, se tiene **11702** artículos que deberían enriquecerse para no perder datos relevantes.

In [61]:
# Nulos en columnas author y affiliation
df_articles[["author", "affiliation"]].isnull().all(axis=1).sum()

np.int64(51)

Nótese que los artículos que no incluyen `author` tampoco incluyen `affiliation`.

In [62]:
df_articles[df_articles["dc:title"].isnull()]

,dc:identifier,dc:title,dc:description,authkeywords,citedby-count,prism:coverDate,affiliation,author
48202,SCOPUS_ID:85184927777,NaN,NaN,NaN,0,2023-01-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [63]:
df_articles[df_articles["affiliation"].isnull()].head()

,dc:identifier,dc:title,dc:description,authkeywords,citedby-count,prism:coverDate,affiliation,author
1717,SCOPUS_ID:85189047082,Violence against women and justice from women ...,This his paper analyzes the new forms of viole...,Amazon | emancipatory legal pluralism | violen...,2,2020-07-01,NaN,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
5343,SCOPUS_ID:85085183378,Raising intensity and modeling the process of ...,The key and most energy-consuming process in o...,Extraction | Inulin | Mass transfer | Technolo...,0,2020-01-01,NaN,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
6239,SCOPUS_ID:85177679292,"Between Legality and Legitimacy, a proposal fo...",The purpose of this article is to question whe...,Humanitarian Intervention | International Cour...,1,2019-12-20,NaN,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
9575,SCOPUS_ID:85083410147,Political power in Ecuador,This book is a study of politics and the chang...,NaN,1,2019-01-01,NaN,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
9939,SCOPUS_ID:85076720229,The need for a conventional conflict doctrine ...,Defense notions in South America have traditio...,Armed forces | Chile | Defense | Doctrine | Ec...,0,2019-01-01,NaN,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [64]:
df_articles[df_articles["author"].isnull()]

,dc:identifier,dc:title,dc:description,authkeywords,citedby-count,prism:coverDate,affiliation,author
41436,SCOPUS_ID:85200903905,¿Qué es a Critical american Language Praxis?,NaN,NaN,0,2024-01-01,NaN,NaN
59178,SCOPUS_ID:85099053256,Non-pecuniary Effects of Migration Inflows to ...,The migration phenomenon has important implica...,C01 | D60 | Developing countries | F22 | Gende...,2,2021-12-01,NaN,NaN
59179,SCOPUS_ID:85098942106,Thyroid cancer overdiagnosis and overtreatment...,Background: In contrast to the rapid increase ...,Ecuador | Histopathology | Latin-America | Out...,23,2021-12-01,NaN,NaN
59180,SCOPUS_ID:85098888718,"Knowledge, attitudes and practices on influenz...",Background: Vaccination is the most effective ...,Ecuador | Health providers | Influenza | Influ...,21,2021-12-01,NaN,NaN
59181,SCOPUS_ID:85098881417,Leptospira in river and soil in a highly endem...,Background: Leptospira are shed into the envir...,Environmental detection of Leptospira | Epidem...,25,2021-12-01,NaN,NaN
59182,SCOPUS_ID:85098781022,"Caretaker knowledge, attitudes, and practices ...",Background: The rapid spread of extended-spect...,Antibiotic resistance | attitudes and practice...,8,2021-12-01,NaN,NaN
59183,SCOPUS_ID:85098672850,Author Correction: Tree mode of death and mort...,The original version of this Article contained...,NaN,0,2021-12-01,NaN,NaN
59184,SCOPUS_ID:85098653113,Host relatedness and landscape connectivity sh...,Urban expansion can fundamentally alter wildli...,NaN,22,2021-12-01,NaN,NaN
59185,SCOPUS_ID:85097303182,Beyond Carceral Expansion: Survivors’ Experien...,This article presents empirical findings addre...,Carceral feminism | criminal law | domestic vi...,10,2021-12-01,NaN,NaN
59186,SCOPUS_ID:85094673477,The acoustics of ancient catacombs in Southern...,"The catacombs, burial sites for early Christia...",archeoacoustics | architectural acoustics | Ca...,20,2021-12-01,NaN,NaN


## Artículos Duplicados

Se identifica cuantos artículos duplicados existen en nuestra data, esto por medio del campo `dc:identifier` también llamado Scopus ID.

In [65]:
print(f"Total artículos duplicados: {df_articles["dc:identifier"].duplicated().sum()}")

Total artículos duplicados: 0


# Normalizar Datos

## Garantizar listas en affiliation y author

In [66]:
print("Tipos de datos en la columna 'affiliation':")
print(df_articles["affiliation"].apply(type).value_counts())
print("\nTipos de datos en la columna 'author':")
print(df_articles["author"].apply(type).value_counts())

Tipos de datos en la columna 'affiliation':
affiliation
<class 'list'>     69130
<class 'float'>      262
Name: count, dtype: int64

Tipos de datos en la columna 'author':
author
<class 'list'>     69341
<class 'float'>       51
Name: count, dtype: int64


El resultado previo muestra que todos los registros que presentan valores en `affiliation` y `author` son de tipo lista, mientras que aquellos que no presentan valor son de tipo float (NaN) o nulos.

## Convertir prism:coverDate a tipo datetime

In [67]:
df_articles["prism:coverDate"] = pd.to_datetime(df_articles["prism:coverDate"])
# Orden ascendente
df_articles = df_articles.sort_values("prism:coverDate", ascending=False).reset_index(drop=True)

## Extraer solo valor numerico de scopus_id

In [68]:
df_articles["dc:identifier"] = df_articles["dc:identifier"].str.replace("SCOPUS_ID:", "", regex=False)

## Renombrar atributos

In [69]:
df_articles = df_articles.rename(
    columns={
        "dc:identifier": "scopus_id",
        "dc:title": "title",
        "dc:description": "abstract",
        "authkeywords": "keywords",
        "citedby-count": "cited_by_count",
        "prism:coverDate": "publication_date",
        "affiliation": "affiliations",
        "author": "authors",
    }
)

df_articles.head(3)

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


# Análisis de Completitud

Las inconsistencias que se presentan en esta sección se resuelven, en su mayoría, con llamadas a **`Abstract Retrieval API`** para poder completar información relevante. Por ejemplo, el caso en el que **`Search Scopus API`** limita la longitud de la lista `author` a máximo **100** autores, sin embargo, hay artículos en los que se supera dicho valor, por lo cual la response devuelve una **lista truncada** y posiblemente se pierden autores de afiliaciones ecuatorianas.

In [70]:
incompleted_auth_articles = df_articles[df_articles["authors"].apply(lambda x: isinstance(x, list) and len(x) == 100)]
print(f"Total de artículos con posibles autores faltantes: {len(incompleted_auth_articles)}")

Total de artículos con posibles autores faltantes: 1709


In [71]:
incompleted_auth_articles.head(3)

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
172,105011404413,Transdiagnostic links of reward processing in ...,Objectives: Transdiagnostic models aim to unco...,Assessment | Depression | OCD | Reward respons...,1,2025-12-15,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
180,105010224653,Towards fair decentralized benchmarking of hea...,Computational competitions are the standard fo...,NaN,0,2025-12-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
196,105010693423,Benchmarking empirical severity for the Yale-B...,The Yale-Brown Obsessive Compulsive Scale (Y-B...,Assessment | Obsessive-compulsive disorder | R...,0,2025-12-01,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


El resultado anterior indica que se deben realizar **1709** peticiones a **`Abstract Retrieval API`** para poder completar los posibles autores faltantes en ciertos articulos.

In [72]:
# Guardar scopus_id de articulos con autores incompletos para enriquecer posteriormente
incompleted_auth_articles[["scopus_id"]].to_csv("data/entities/articles_ids_with_incompleted_auths.csv", index=False)

También se debe considerar los casos evidenciados en la subsección [Valores Nulos](#valores-nulos), en la cual se obtuvo que **11702** artículos deberían enriquecerse por falta de datos.

# Procesamiento de Entidades

## 1. Afiliaciones

En la mayoría de los casos, cada artículo incluye el atributo `affiliation` cuyo valor es una **lista** de afiliaciones. Esta lista contiene al menos una afiliación; de lo contrario, el atributo no está presente en el artículo.

Ejemplo de estructura:

```json
"affiliation": [
  {
    "@_fa": "true",
    "affiliation-url": "https://api.elsevier.com/content/affiliation/affiliation_id/60072054",
    "afid": "60072054",
    "affilname": "Escuela Politécnica Nacional",
    "affiliation-city": "Quito",
    "affiliation-country": "Ecuador"
  },
  {
    "@_fa": "true",
    "affiliation-url": "https://api.elsevier.com/content/affiliation/affiliation_id/60002439",
    "afid": "60002439",
    "affilname": "Universidad De Los Andes, Merida",
    "affiliation-city": "Merida",
    "affiliation-country": "Venezuela"
  }
]
```





### Extraer todas las afiliaciones

In [73]:
# Expandir lista: cada afiliacion dentro de la lista pasa a ser una fila
df_affs = (
    df_articles.explode("affiliations")
    .dropna(subset=["affiliations"])
    .reset_index(drop=True)
)

In [74]:
# Convertir cada atributo de la afiliacion a columna
df_affs = pd.json_normalize(df_affs["affiliations"])
df_affs = df_affs.drop(columns=["@_fa", "affiliation-url"], errors="ignore")

print(f"Forma de df_affs: {df_affs.shape}\n")
df_affs.head()

Forma de df_affs: (298771, 4)



,afid,affilname,affiliation-city,affiliation-country
0,60278953,Universidad Bolivariana del Ecuador,Duran,Ecuador
1,60278953,Universidad Bolivariana del Ecuador,Duran,Ecuador
2,60278953,Universidad Bolivariana del Ecuador,Duran,Ecuador
3,133242834,Instituto de Investigación Multidisciplinaria ...,None,Ecuador
4,60108912,Universidad Técnica de Manabí,Portoviejo,Ecuador


In [75]:
# lista con ids de todas las afiliaciones
aff_ids = df_affs["afid"].unique().tolist()
print(f"Total de afiliaciones únicas: {len(aff_ids)}")

Total de afiliaciones únicas: 38255


In [76]:
df_all_affs = df_affs.copy()
df_all_affs = df_all_affs.drop_duplicates(subset=["afid"]).reset_index(drop=True)
print(f"Forma de df_all_affs (afiliaciones únicas): {df_all_affs.shape}\n")
df_all_affs[df_all_affs["affiliation-country"] != "Ecuador"].head()

Forma de df_all_affs (afiliaciones únicas): (38255, 4)



,afid,affilname,affiliation-city,affiliation-country
5,112804617,Ingá University Center UNINGÁ,Maringa,Brazil
10,60055678,Universidad de Cartagena,Cartagena,Colombia
14,60023020,Universitat Autònoma de Barcelona,Cerdanyola del Valles,Spain
17,60015361,Slovak Medical University,Bratislava,Slovakia
18,60015734,Institute of Molecular Biology Slovak Academy ...,Bratislava,Slovakia


In [77]:
df_all_affs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38255 entries, 0 to 38254
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   afid                 38255 non-null  object
 1   affilname            38255 non-null  object
 2   affiliation-city     33366 non-null  object
 3   affiliation-country  37910 non-null  object
dtypes: object(4)
memory usage: 1.2+ MB


In [78]:
df_all_affs.to_csv("all_affiliations.csv", index=False)

### Filtrar para Ecuador y eliminar duplicados

In [79]:
df_affs_ec = (
    df_affs[df_affs["affiliation-country"] == "Ecuador"]
    .drop_duplicates(subset=["afid"])
    .reset_index(drop=True)
)

print(f"Forma de df_affs_ec: {df_affs_ec.shape}\n")
df_affs_ec.head()

Forma de df_affs_ec: (8143, 4)



,afid,affilname,affiliation-city,affiliation-country
0,60278953,Universidad Bolivariana del Ecuador,Duran,Ecuador
1,133242834,Instituto de Investigación Multidisciplinaria ...,None,Ecuador
2,60108912,Universidad Técnica de Manabí,Portoviejo,Ecuador
3,131940052,Unversidad Estatal de Milagro,Milagro,Ecuador
4,60072064,Universidad Técnica Particular de Loja,Loja,Ecuador


In [80]:
# Valores nulos por columna
df_affs_ec.isnull().sum()

afid                      0
affilname                 0
affiliation-city       2355
affiliation-country       0
dtype: int64

### Exportar afiliaciones

In [81]:
afids_ec = set(df_affs_ec["afid"])
df_affs_ec.to_csv("data/entities/afiliaciones_ecuador.csv", index=False)

## 2. Autores

### Extraer todos los autores

In [27]:
df_auths = df_articles.explode("authors").dropna(subset=["authors"]).reset_index(drop=True)

In [28]:
df_auths = pd.json_normalize(df_auths["authors"])
df_auths = df_auths.drop(columns=["@_fa", "@seq", "author-url"], errors="ignore")

print(f"Forma de df_auths: {df_auths.shape}\n")
df_auths.head()

Forma de df_auths: (543934, 7)



,authid,authname,surname,given-name,initials,afid,orcid
0,58611443600,Jaramillo Castellón L.,Jaramillo Castellón,Lourdes,L.,"[{'@_fa': 'true', '$': '60278953'}]",NaN
1,59711816000,Cedeño Sánchez L.V.,Cedeño Sánchez,Laly Viviana,L.V.,"[{'@_fa': 'true', '$': '60278953'}]",NaN
2,59326400900,Guerra Iglesias S.,Guerra Iglesias,Sonia,S.,"[{'@_fa': 'true', '$': '60278953'}]",NaN
3,59704488800,Bodero Arizaga L.D.C.,Bodero Arizaga,Lorena Del Carmen,L.D.C.,"[{'@_fa': 'true', '$': '60278953'}]",NaN
4,60130763800,González R.E.A.,González,Rously Eedyah Atencio,R.E.A.,"[{'@_fa': 'true', '$': '60278953'}]",NaN


### Extraer afids de cada autor

In [29]:
def extract_afids(auth_list):
    if not isinstance(auth_list, list):
        return []
    
    return [item.get("$") for item in auth_list if isinstance(item, dict) and "$" in item]

In [30]:
df_auths["afid"] = df_auths["afid"].apply(extract_afids)
print(f"Forma de df_auths: {df_auths.shape}\n")
df_auths

Forma de df_auths: (543934, 7)



,authid,authname,surname,given-name,initials,afid,orcid
0,58611443600,Jaramillo Castellón L.,Jaramillo Castellón,Lourdes,L.,[60278953],NaN
1,59711816000,Cedeño Sánchez L.V.,Cedeño Sánchez,Laly Viviana,L.V.,[60278953],NaN
2,59326400900,Guerra Iglesias S.,Guerra Iglesias,Sonia,S.,[60278953],NaN
3,59704488800,Bodero Arizaga L.D.C.,Bodero Arizaga,Lorena Del Carmen,L.D.C.,[60278953],NaN
4,60130763800,González R.E.A.,González,Rously Eedyah Atencio,R.E.A.,[60278953],NaN
...,...,...,...,...,...,...,...
543929,56852172200,Rojas F.,Rojas,F.,F.,[60072042],NaN
543930,56852475000,Morengo J.,Morengo,J. Tanca,J.T.,[60072042],NaN
543931,56910946000,Royer E.,Royer,E. Ray,E.R.,[115872020],NaN
543932,56910852500,Connor M.,Connor,M. E.,M.E.,[115871777],NaN


### Filtrar autores con afiliaciones ecuatorianas

In [31]:
def filter_afids_ec(lst):
    if not isinstance(lst, list):
        return []
    return [x for x in lst if x in afids_ec]

In [32]:
df_auths["afid"] = df_auths["afid"].apply(filter_afids_ec)
df_auths_ec = df_auths[df_auths["afid"].apply(len) > 0].copy()
df_auths_ec

,authid,authname,surname,given-name,initials,afid,orcid
0,58611443600,Jaramillo Castellón L.,Jaramillo Castellón,Lourdes,L.,[60278953],NaN
1,59711816000,Cedeño Sánchez L.V.,Cedeño Sánchez,Laly Viviana,L.V.,[60278953],NaN
2,59326400900,Guerra Iglesias S.,Guerra Iglesias,Sonia,S.,[60278953],NaN
3,59704488800,Bodero Arizaga L.D.C.,Bodero Arizaga,Lorena Del Carmen,L.D.C.,[60278953],NaN
4,60130763800,González R.E.A.,González,Rously Eedyah Atencio,R.E.A.,[60278953],NaN
...,...,...,...,...,...,...,...
543928,36511972300,Pachano A.,Pachano,Abelardo,A.,[109557708],NaN
543929,56852172200,Rojas F.,Rojas,F.,F.,[60072042],NaN
543930,56852475000,Morengo J.,Morengo,J. Tanca,J.T.,[60072042],NaN
543931,56910946000,Royer E.,Royer,E. Ray,E.R.,[115872020],NaN


### Merge de afids por autor

In [33]:
def first_not_null(series):
    series = series.dropna()
    return series.iloc[0] if not series.empty else np.nan

In [34]:
def merge_afid(s):
    afids = []
    for v in s.dropna():
        if isinstance(v, list):
            afids.extend(v)
    return sorted(set(afids))

In [35]:
df_auths_ec = (
    df_auths_ec.groupby("authid")
    .agg(
        {
            "authname": "first",
            "surname": "first",
            "given-name": "first",
            "initials": "first",
            "orcid": first_not_null,
            "afid": lambda x: sorted(set(a for lst in x for a in lst)),
        }
    )
    .reset_index()
)

print(f"Autores únicos de Ecuador: {len(df_auths_ec)}")
df_auths_ec.head()

Autores únicos de Ecuador: 65547


,authid,authname,surname,given-name,initials,orcid,afid
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"[101703861, 60104598]"
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,[60104598]
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"[108330837, 117939271, 118823220, 133222327, 6..."
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,[60072059]
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,[60072063]


### Exportar Autores

In [36]:
authids_ec = set(df_auths_ec["authid"])
df_auths_ec.to_csv("data/entities/autores_ecuador.csv", index=False)

## 3. Articulos

### Conteo de papers por afiliacion (fragmentacion)

In [82]:
# funcion para extraer affilname de las afiliaciones
def extract_affids(aff_list):
    if not isinstance(aff_list, list):
        return []
    
    return [
        aff.get("afid")
        for aff in aff_list
        if isinstance(aff, dict) and aff.get("afid") in aff_ids
    ]

In [83]:
df_articles_full = df_articles.copy()

In [84]:
df_articles_full["affiliations"] = df_articles["affiliations"].apply(extract_affids)
df_articles_full.head()

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,[60278953],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,[60278953],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,"[60278953, 133242834]","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,digital teaching skills | disability | higher ...,0,2026-07-16,[60108912],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
4,105015199172,Key Factors for the Success of Microenterprise...,Introduction: This article examines entreprene...,business management | competencies | competiti...,0,2026-07-16,[131940052],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [85]:
aff = df_articles_full.explode('affiliations').dropna(subset=['affiliations'])
print(f"Forma de aff: {aff.shape}\n")
aff.head()

Forma de aff: (298771, 8)



,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,60278953,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,60278953,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,60278953,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,133242834,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,digital teaching skills | disability | higher ...,0,2026-07-16,60108912,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [86]:
counts_full = (aff.groupby('affiliations')['scopus_id']
            .nunique()
            .rename('n_articulos')
            .reset_index())

In [87]:
bins = [0, 1, 5, 50, np.inf]
labels = ['1', '2-5', '6-50', '>50']
counts_full['rango'] = pd.cut(counts_full['n_articulos'], bins=bins, labels=labels, right=True)

resumen = (counts_full['rango']
           .value_counts()
           .reindex(labels, fill_value=0)
           .rename_axis('rango')
           .reset_index(name='n_afiliaciones'))

resumen

,rango,n_afiliaciones
0,1,23304
1,2-5,9569
2,6-50,4661
3,>50,721


total = 38,255

In [31]:
# Se requiere obtener estadisticas de las afiliaciones por artículo, es decir, cuantas afilaiciones cuentan con un solo paper, y cuantas con varios
# Crear un DataFrame para contar la cantidad de afiliaciones con cierta cantidad de articulos (fragmentar en 1, 2-5, 6-50, >50)
article_aff_counts = df_articles.explode("affiliations").groupby("affiliations").size().reset_index(name="article_count")
article_aff_counts.head()

,affiliations,article_count
0,"""CRECIENDO""",1
1,"""Dr. Arnoldo Gabaldón"" High Studies Institute",1
2,"""Dr. Teodoro Maldonado Carbo"" Hospital",1
3,"""Enrique José Varona” Pedagogical Sciences Uni...",1
4,"""Grad. Miguel Iturralde""",2


In [32]:
article_aff_counts.shape

(34716, 2)

### Filtrar afiliaciones y autores ecuatorianos

In [27]:
df_articles.head()

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,digital teaching skills | disability | higher ...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
4,105015199172,Key Factors for the Success of Microenterprise...,Introduction: This article examines entreprene...,business management | competencies | competiti...,0,2026-07-16,"[{'@_fa': 'true', 'affiliation-url': 'https://...","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [88]:
# funcion para extraer affilname de las afiliaciones de ecuador
def extract_ec_affids(aff_list):
    if not isinstance(aff_list, list):
        return []
    
    return [
        aff.get("afid")
        for aff in aff_list
        if isinstance(aff, dict) and aff.get("afid") in afids_ec
    ]

In [89]:
df_articles_ec = df_articles.copy()

In [90]:
df_articles_ec["affiliations"] = df_articles_ec["affiliations"].apply(extract_ec_affids)
df_articles_ec.head()

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,[60278953],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,[60278953],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,"[60278953, 133242834]","[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,digital teaching skills | disability | higher ...,0,2026-07-16,[60108912],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
4,105015199172,Key Factors for the Success of Microenterprise...,Introduction: This article examines entreprene...,business management | competencies | competiti...,0,2026-07-16,[131940052],"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [91]:
aff_ec = df_articles_ec.explode('affiliations').dropna(subset=['affiliations'])
aff_ec.head()

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,60278953,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,60278953,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,60278953,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,133242834,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,digital teaching skills | disability | higher ...,0,2026-07-16,60108912,"[{'@_fa': 'true', '@seq': '1', 'author-url': '..."


In [92]:
counts = (aff_ec.groupby('affiliations')['scopus_id']
            .nunique()
            .rename('n_articulos')
            .reset_index())

In [93]:
bins = [0, 1, 5, 50, np.inf]
labels = ['1', '2-5', '6-50', '>50']
counts['rango'] = pd.cut(counts['n_articulos'], bins=bins, labels=labels, right=True)

resumen_ec = (counts['rango']
           .value_counts()
           .reindex(labels, fill_value=0)
           .rename_axis('rango')
           .reset_index(name='n_afiliaciones'))

resumen_ec

,rango,n_afiliaciones
0,1,5462
1,2-5,1957
2,6-50,633
3,>50,91


total = 8143

*before*
1: 23304 (60.92%)
2-5: 9569 (25.01%)
6-50: 4661 (12.18%)
>50: 721 (1.88%)
total: 38,255
*after*
1: 5462 (67.08%)
2-5: 1957 (24.03%)
6-50: 633 (7.77%)
>50: 91 (1.12%)
total: 8143

razon por la que no es buena idea agrupar por affilname es porque hay afiliaciones con el mismo nombre pero difieren en afid.

In [30]:
df_articles.to_csv("articulos_new.csv", index=False)

In [39]:
def filter_authids_ec(author_list):
    if not isinstance(author_list, list):
        return []
    ids = []
    for a in author_list:
        if isinstance(a, dict) and "authid" in a:
            aid = a["authid"]
            if aid in authids_ec:
                ids.append(aid)
    return ids

In [40]:
df_articles["authors"] = df_articles["authors"].apply(filter_authids_ec)

In [41]:
# Convertir listas vacias a NaN
df_articles["affiliations"] = df_articles["affiliations"].apply(lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x)
df_articles["authors"] = df_articles["authors"].apply(lambda x: np.nan if isinstance(x, list) and len(x) == 0 else x)
df_articles.iloc[[0, 1, -2, -1]]

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,NaN,"[58611443600, 59711816000]"
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,NaN,"[59326400900, 59704488800]"
69390,84944670009,Yellow fever control in ecuador: Final report,NaN,NaN,0,1920-10-30,NaN,[56910852500]
69391,84944670184,Yellow fever control in ecuador: Preliminary r...,NaN,NaN,4,1920-03-06,NaN,[56910852500]


Hay metadata erronea, por ejemplo el caso del articulo 0000187288 cuya afilaicion es de EEUU pero la metadata en Abstract API tiene la etiqueta de Ecuador, por ende esa relación es erronea.

### Exportar Artículos

In [44]:
df_articles.to_csv("data/entities/articulos_ecuador.csv", index=False)

## 4. Topics

### Convertir keywords a lista de topics

In [45]:
df_topics = df_articles.copy()
df_topics = df_topics.dropna(subset=["keywords"])
df_topics["topic"] = df_topics["keywords"].str.split("|")
df_topics.head(3)

,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors,topic
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,[60278953],"[58611443600, 59711816000]","[clinical communication , communicative compe..."
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,[60278953],"[59326400900, 59704488800]","[competence , conception , educational inclu..."
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,"[60278953, 133242834]","[60130763800, 60130890600]","[factor analysis , higher education , Hybrid..."


### Extraer todos los topics

In [46]:
df_topics = df_topics.explode("topic")
df_topics["topic"] = df_topics["topic"].str.strip().str.lower()

In [47]:
topics = (
    df_topics[["topic"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f"Número de topics únicos: {len(topics)}")
topics.head()

Número de topics únicos: 122124


,topic
0,clinical communication
1,communicative competencies
2,competency-based approach
3,dental education
4,english for professional purposes (epp)


### Exportar topics

In [48]:
topics.to_csv("data/entities/topics.csv", index=False)

## 5. Subject Areas

In [49]:
import ast

df_subject_areas = pd.read_csv("data/entities/autores_ecuador_enriquecido.csv")

In [50]:

def parse_subject_areas(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return [x]

# 1. Parsear subject-areas → lista
df_subject_areas["subject_area"] = (
    df_subject_areas["subject-areas"]
    .apply(parse_subject_areas)
)

# 2. Explode correctamente (sobre subject_area)
df_subject_areas = df_subject_areas.explode("subject_area")

# 3. Limpieza
df_subject_areas["subject_area"] = df_subject_areas["subject_area"].astype(str).str.strip()
df_subject_areas = df_subject_areas[
    df_subject_areas["subject_area"].notna() &
    (df_subject_areas["subject_area"] != "") &
    (df_subject_areas["subject_area"].str.lower() != "nan")
]


In [51]:
df_author_area = (
    df_subject_areas[["authid", "subject_area"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_author_area.head()

,authid,subject_area
0,10039323700,Computer Science Applications
1,10039323700,Astronomy and Astrophysics
2,10039323700,Medicine (all)
3,10039323700,Analytical Chemistry
4,10039323700,Modeling and Simulation


In [52]:
df_subject_area_entity = (
    df_author_area[["subject_area"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

df_subject_area_entity.head()

,subject_area
0,Computer Science Applications
1,Astronomy and Astrophysics
2,Medicine (all)
3,Analytical Chemistry
4,Modeling and Simulation


In [54]:
df_author_area.to_csv("data/csv-neo/author_area.csv", index=False)
df_subject_area_entity.to_csv("data/csv-neo/subject_areas.csv", index=False)

# Construir Relaciones

## Author_Affiliation

In [55]:
df_authors_ec = pd.read_csv(
    "data/entities/autores_ecuador_enriquecido.csv",
    dtype={
        "authid": "object",
        "orcid": "object",
    },
)
print(f"Forma de df_authors_ec: {df_authors_ec.shape}\n")
df_authors_ec.head()

Forma de df_authors_ec: (65547, 9)



,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"['101703861', '60104598']",20.0,"['Computer Science Applications', 'Astronomy a..."
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,['60104598'],2.0,"['Computational Theory and Mathematics', 'Comp..."
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"['108330837', '117939271', '118823220', '13322...",11.0,"['Ecology', 'Business, Management and Accounti..."
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,['60072059'],9.0,"['Immunology', 'Medicine (all)', 'Earth and Pl..."
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,['60072063'],2.0,['Plant Science']


In [56]:
df_authors_ec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 65547 entries, 0 to 65546
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   authid         65547 non-null  object 
 1   authname       65547 non-null  object 
 2   surname        65547 non-null  object 
 3   given-name     65519 non-null  object 
 4   initials       65519 non-null  object 
 5   orcid          23615 non-null  object 
 6   afid           65547 non-null  object 
 7   h-index        65525 non-null  float64
 8   subject-areas  65492 non-null  object 
dtypes: float64(1), object(8)
memory usage: 4.5+ MB


In [57]:
df_authors_ec.isna().sum()

authid               0
authname             0
surname              0
given-name          28
initials            28
orcid            41932
afid                 0
h-index             22
subject-areas       55
dtype: int64

In [58]:
import ast

df_authors_ec["afid"] = df_authors_ec["afid"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

df_authors_ec["subject-areas"] = df_authors_ec["subject-areas"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [59]:
df_authors_ec.head()

,authid,authname,surname,given-name,initials,orcid,afid,h-index,subject-areas
0,10039323700,Apolinário J.,Apolinário,J. A.,J.A.,NaN,"[101703861, 60104598]",20.0,"[Computer Science Applications, Astronomy and ..."
1,10040479200,Almeida C.,Almeida,Carlos,C.,NaN,[60104598],2.0,"[Computational Theory and Mathematics, Computa..."
2,10040712400,Carrión V.,Carrión,Víctor,V.,0009-0007-5806-5471,"[108330837, 117939271, 118823220, 133222327, 6...",11.0,"[Ecology, Business, Management and Accounting ..."
3,10043217500,Friedman R.,Friedman,Risa,R.,NaN,[60072059],9.0,"[Immunology, Medicine (all), Earth and Planeta..."
4,10240259200,Yandún S.,Yandún,Santiago,S.,NaN,[60072063],2.0,[Plant Science]


In [60]:
affiliations_df = (
    df_authors_ec[["afid"]]
    .explode("afid")
    .dropna()
    .drop_duplicates()
)
print(f"Forma de affiliations_df: {affiliations_df.shape}\n")
affiliations_df.head()

Forma de affiliations_df: (8143, 1)



,afid
0,101703861
0,60104598
2,108330837
2,117939271
2,118823220


In [61]:
subject_areas_df = (
    df_authors_ec[["subject-areas"]]
    .explode("subject-areas")
    .dropna()
    .drop_duplicates()
)
print(f"Forma de subject_areas_df: {subject_areas_df.shape}\n")
subject_areas_df.head()

Forma de subject_areas_df: (332, 1)



,subject-areas
0,Computer Science Applications
0,Astronomy and Astrophysics
0,Medicine (all)
0,Analytical Chemistry
0,Modeling and Simulation


In [62]:
df_author_affiliation = (
    df_authors_ec[["authid", "afid"]]
    .explode("afid")
    .dropna()
)
df_author_affiliation

,authid,afid
0,10039323700,101703861
0,10039323700,60104598
1,10040479200,60104598
2,10040712400,108330837
2,10040712400,117939271
...,...,...
65546,9942669300,101028104
65546,9942669300,106715575
65546,9942669300,114795154
65546,9942669300,125028819


In [63]:
df_author_affiliation[df_author_affiliation["authid"] == "58697225500"]

,authid,afid
44836,58697225500,118470467
44836,58697225500,60072059


In [64]:
df_author_subject = (
    df_authors_ec[["authid", "subject-areas"]]
    .explode("subject-areas")
    .dropna()
)
print(f"Forma de df_author_subject: {df_author_subject.shape}\n")
df_author_subject.head()

Forma de df_author_subject: (379764, 2)



,authid,subject-areas
0,10039323700,Computer Science Applications
0,10039323700,Astronomy and Astrophysics
0,10039323700,Medicine (all)
0,10039323700,Analytical Chemistry
0,10039323700,Modeling and Simulation


In [65]:
df_author_affiliation.to_csv("data/csv-neo/autor_afiliacion.csv", index=False)

In [66]:
subject_areas_df.to_csv("data/csv-neo/subject_areas.csv", index=False)

In [67]:
df_auths_neo = df_authors_ec.drop(columns=["afid", "subject-areas", "h-index"])
df_auths_neo.to_csv("data/csv-neo/autores_neo.csv", index=False)

## Author_Article

In [68]:
df_articles_ec = pd.read_csv(
    "data/entities/articulos_ecuador.csv",
    
)
print(f"Forma de df_articles_ec: {df_articles_ec.shape}\n")
df_articles_ec.head()

Forma de df_articles_ec: (69392, 8)



,scopus_id,title,abstract,keywords,cited_by_count,publication_date,affiliations,authors
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,clinical communication | communicative compete...,0,2026-07-16,['60278953'],"['58611443600', '59711816000']"
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,competence | conception | educational inclusio...,0,2026-07-16,['60278953'],"['59326400900', '59704488800']"
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,factor analysis | higher education | Hybrid mo...,0,2026-07-16,"['60278953', '133242834']","['60130763800', '60130890600']"
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,digital teaching skills | disability | higher ...,0,2026-07-16,['60108912'],"['36486889700', '60089589800', '57208549638']"
4,105015199172,Key Factors for the Success of Microenterprise...,Introduction: This article examines entreprene...,business management | competencies | competiti...,0,2026-07-16,['131940052'],"['59188181600', '60086443500', '60086641500', ..."


In [69]:
import ast

df_articles_ec["affiliations"] = df_articles_ec["affiliations"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

df_articles_ec["authors"] = df_articles_ec["authors"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

In [70]:
df_author_article = (
    df_articles_ec[["scopus_id", "authors"]]
    .explode("authors")
    .dropna()
    .rename(columns={"authors": "authid"})
)
print(f"Forma de df_author_article: {df_author_article.shape}\n")
df_author_article.head()

Forma de df_author_article: (198718, 2)



,scopus_id,authid
0,105018508936,58611443600
0,105018508936,59711816000
1,105018495934,59326400900
1,105018495934,59704488800
2,105018109024,60130763800


In [71]:
df_author_article.to_csv("data/csv-neo/autor_articulo.csv", index=False)

## Article_Affiliation

In [72]:
df_article_aff = (
    df_articles_ec[["scopus_id", "affiliations"]]
    .explode("affiliations")
    .dropna()
    .rename(columns={"affiliations": "affid"})
)
print(f"Forma de df_article_aff: {df_article_aff.shape}\n")
df_article_aff.head()

Forma de df_article_aff: (95478, 2)



,scopus_id,affid
0,105018508936,60278953
1,105018495934,60278953
2,105018109024,60278953
2,105018109024,133242834
3,105015321180,60108912


In [73]:
df_article_aff.to_csv("data/csv-neo/articulo_afiliacion.csv", index=False)

In [74]:
df_arts_neo = df_articles_ec.drop(columns=["authors", "affiliations", "keywords"])
df_arts_neo.to_csv("data/csv-neo/articulos_neo.csv", index=False)

In [75]:
df_arts_neo.head()

,scopus_id,title,abstract,cited_by_count,publication_date
0,105018508936,Integration of English for professional purpos...,Introduction: Current dental education faces t...,0,2026-07-16
1,105018495934,Integration of knowledge and multi-approaches:...,Introduction: The multidimensionality of study...,0,2026-07-16
2,105018109024,Indicators of confidence in hybrid education a...,Introduction: The transition to a hybrid modal...,0,2026-07-16
3,105015321180,Digital teaching skills and disability: a prop...,Introduction: Digital teaching skills are fund...,0,2026-07-16
4,105015199172,Key Factors for the Success of Microenterprise...,Introduction: This article examines entreprene...,0,2026-07-16


## Topic_Article

In [76]:
topic_article = df_topics[[
    "topic",
    "scopus_id"
]].drop_duplicates()

topic_article.columns = ["topic", "article_id"]
topic_article.head()

,topic,article_id
0,clinical communication,105018508936
0,communicative competencies,105018508936
0,competency-based approach,105018508936
0,dental education,105018508936
0,english for professional purposes (epp),105018508936


In [77]:
topic_article.to_csv("data/entities/topic_article.csv", index=False)

## Topic_Author

In [78]:
df_topics = df_topics.explode("authors")
df_topics = df_topics.rename(columns={"authors": "author_id"})

In [79]:
topic_author = df_topics[[
    "topic",
    "author_id"
]].drop_duplicates()

topic_author.head()

,topic,author_id
0,clinical communication,58611443600
0,clinical communication,59711816000
0,communicative competencies,58611443600
0,communicative competencies,59711816000
0,competency-based approach,58611443600


In [80]:
topic_author.to_csv("data/csv-neo/topic_author.csv", index=False)